# 01 — Preparação de dados, anonimização e curadoria

**Tech Challenge Fase 3 — MedFlow AI**

Este notebook demonstra, de forma reprodutível, as três etapas obrigatórias de preparação de dados
exigidas pelo enunciado: **preprocessing**, **anonimização** e **curadoria**.

Cada seção responde a uma pergunta explícita e termina com interpretação — não são gráficos soltos.

> ⚠️ Todos os dados usados aqui são **sintéticos**. Nenhum dado real de paciente é utilizado.

In [ ]:
# Execução local ou no Colab. No Colab, descomente o bloco de clone.
# !git clone https://github.com/NirtonAfonso/tech-challenge-fase3-medflow-ai.git
# %cd tech-challenge-fase3-medflow-ai
# !pip install -q -r requirements.txt

import sys, pathlib
RAIZ = pathlib.Path.cwd()
while not (RAIZ / "src" / "medflow_ai").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
print("Raiz do projeto:", RAIZ)

## 1. Qual é o corpus institucional e de onde ele vem?

O corpus são 15 documentos Markdown versionados em `data/synthetic/protocols/`, representando
protocolos clínicos, procedimentos internos, modelos de laudo e receita e perguntas frequentes de
médicos — exatamente as categorias que o enunciado pede para o fine-tuning.

Manter o corpus em texto (e não em PDF binário) torna cada alteração rastreável pelo próprio `git`.

In [ ]:
from medflow_ai.data.corpus import load_corpus

documentos = load_corpus()
print(f"{len(documentos)} documentos | {sum(len(d.sections) for d in documentos)} seções\n")
for doc in documentos:
    print(f"{doc.doc_id:14s} v{doc.version:5s} {doc.doc_type:16s} {len(doc.sections):2d} seções  {doc.title[:52]}")

In [ ]:
# Origem declarada de cada documento: exigência de rastreabilidade.
for doc in documentos[:3]:
    print(f"{doc.doc_id}: {doc.metadata['origem']}\n")

**Interpretação.** Os 15 documentos cobrem 6 especialidades e 4 tipos documentais. Nenhum é
documento oficial: todos declaram explicitamente a origem sintética, o que evita atribuir ao
Ministério da Saúde um conteúdo que não é dele.

**Limitação.** Um hospital real teria centenas de protocolos; o corpus aqui é dimensionado para
caber no repositório e permitir avaliação reprodutível em CI.

## 2. A anonimização realmente remove identificadores? (antes / depois)

Esta é a evidência central do requisito de anonimização. Usamos um registro do prontuário sintético
que contém, propositalmente, identificadores diretos falsos.

In [ ]:
from medflow_ai.database.ingest import build_synthetic_database
from medflow_ai.database.repository import PatientRepository

build_synthetic_database(n_patients=40)
repo = PatientRepository()
bruto = repo.raw_record("P-DEMO-0001")

print("=== ANTES (registro bruto, com identificadores diretos) ===")
for chave, valor in bruto.items():
    print(f"  {chave:12s}: {valor}")

In [ ]:
from medflow_ai.data.anonymization import anonymize_record

anonimizado, relatorio = anonymize_record(bruto)

print("=== DEPOIS (registro anonimizado) ===")
for chave, valor in anonimizado.items():
    print(f"  {chave:12s}: {valor}")

print("\n=== RELATÓRIO DE ANONIMIZAÇÃO ===")
import json; print(json.dumps(relatorio.to_dict(), ensure_ascii=False, indent=2))

In [ ]:
from medflow_ai.data.anonymization import anonymize_text

texto = (
    "Paciente: Maria da Silva Souza, CPF 123.456.789-00, CNS 700 5049 3417 8563, "
    "e-mail maria.souza@exemplo.com.br, telefone (11) 98765-4321, residente na "
    "Rua das Acácias, nº 120, CEP 01310-100. Data de nascimento: 12/03/1975. "
    "Prontuário nº 4457821. Atendida pelo Dr. Carlos Andrade, CRM/SP 123456. "
    "Diagnóstico: hipotireoidismo primário, TSH 8,4 mUI/L."
)
limpo, rel = anonymize_text(texto, redact_dates=True)
print("ANTES :", texto, "\n")
print("DEPOIS:", limpo, "\n")
print("Identificadores encontrados por tipo:", rel.counts_by_kind)

**Interpretação.** Oito classes de identificador direto são removidas; o conteúdo clínico
(`TSH 8,4 mUI/L`, `hipotireoidismo primário`) é integralmente preservado. A data de nascimento é
*transformada* em idade e faixa etária, e não simplesmente apagada — preservando utilidade clínica
com menor risco de reidentificação.

**Trade-off.** O reconhecimento de nome depende de um rótulo explícito (`Paciente:`, `Dr.`). Um nome
solto no meio da frase não é capturado. A alternativa (NER treinado) traria falsos positivos sobre
termos clínicos e uma dependência pesada; a escolha aqui privilegia previsibilidade e auditabilidade.

**Limitação.** Cidade/UF permanecem no texto (quase-identificador). Em produção, seriam generalizados
para região.

## 3. O que a curadoria remove, e quanto?

In [ ]:
from medflow_ai.fine_tuning.dataset import (
    generate_examples, anonymize_examples, curate, DatasetStats, split_by_document, HELD_OUT_DOCUMENTS
)

brutos = generate_examples()
print(f"Exemplos gerados: {len(brutos)}")

# Antes da anonimização: quantos exemplos ainda carregam PII?
from medflow_ai.data.anonymization import contains_pii
com_pii = [e for e in brutos if contains_pii(e.instruction) or contains_pii(e.output)]
print(f"Exemplos com PII antes da anonimização: {len(com_pii)}")

limpos, rel_anon = anonymize_examples(brutos)
print(f"Identificadores removidos: {rel_anon.counts_by_kind}")

stats = DatasetStats(gerados=len(limpos))
curados = curate(limpos, stats)
import json; print("\nEstatísticas de curadoria:"); print(json.dumps(stats.to_dict(), ensure_ascii=False, indent=2))

In [ ]:
import collections
familias = collections.Counter(e.familia for e in curados)
largura = max(familias.values())
for familia, total in familias.most_common():
    print(f"{familia:20s} {total:3d} {'█' * int(40 * total / largura)}")

**Interpretação.** A curadoria aplica seis filtros (duplicidade, comprimento mínimo e máximo,
idioma, PII residual). Nenhum exemplo é descartado por PII **depois** da anonimização — o que é o
resultado desejado: o filtro existe como rede de segurança, não como principal mecanismo.

**Observação honesta.** O volume final (~137 exemplos) é pequeno para fine-tuning. É suficiente para
demonstrar adaptação de **formato e comportamento**, e insuficiente para ensinar conhecimento clínico
novo — o que é exatamente a divisão de responsabilidades adotada no projeto (conhecimento fica no RAG).

## 4. Como o split evita data leakage?

Regra adotada: **split por documento**, não por pergunta. Três documentos inteiros são reservados ao
teste. Se dividíssemos perguntas aleatoriamente, uma pergunta de treino e outra de teste poderiam vir
da mesma seção, tornando o benchmark trivial.

In [ ]:
splits = split_by_document(curados)
for nome, itens in splits.items():
    docs = sorted({e.doc_id for e in itens})
    print(f"{nome:12s} {len(itens):3d} exemplos | {len(docs)} documentos")

treino = {e.doc_id for e in splits["train"]} | {e.doc_id for e in splits["validation"]}
teste = {e.doc_id for e in splits["test"]}
print(f"\nDocumentos reservados ao teste: {sorted(teste)}")
print(f"Interseção treino ∩ teste: {sorted(treino & teste) or 'VAZIA ✅'}")

In [ ]:
from medflow_ai.fine_tuning.dataset import build_sft_dataset

splits, stats, caminhos = build_sft_dataset()
print("Arquivos gerados:")
for nome, caminho in caminhos.items():
    print(f"  {nome:12s} {caminho}")

manifesto = json.loads(caminhos["manifest"].read_text(encoding="utf-8"))
print("\nManifesto (trecho):")
print(json.dumps({k: manifesto[k] for k in ("seed", "held_out_documents", "splits")},
                 ensure_ascii=False, indent=2))

In [ ]:
# Exemplo final, no formato de chat consumido pelo SFTTrainer
exemplo = splits["train"][0].to_chat()
for mensagem in exemplo["messages"]:
    print(f"--- {mensagem['role'].upper()} ---")
    print(mensagem["content"][:600])
    print()

**Interpretação.** A interseção entre documentos de treino e de teste é vazia por construção, e o
manifesto registra seed, contagens e *fingerprints* de cada exemplo — qualquer pessoa pode verificar
que o split não mudou entre uma execução e outra.

**Limitação declarada.** O split por documento controla o leakage do *fine-tuning*. Ele não elimina a
limitação inerente da avaliação de RAG, em que o mesmo corpus é indexado e consultado; isso é medido e
declarado separadamente no notebook 03.

## 5. Conclusão do notebook

| Pergunta | Resposta |
|---|---|
| A anonimização funciona? | Sim: 8 classes de identificador removidas, conteúdo clínico preservado |
| A curadoria é mensurável? | Sim: estatísticas antes/depois versionadas no manifesto |
| Há controle de leakage? | Sim: split por documento, com interseção vazia verificada |
| O dataset é suficiente? | Para formato/comportamento sim; para conhecimento clínico não (isso é papel do RAG) |

Próximo notebook: `02_fine_tuning_qlora.ipynb` (execução em GPU no Google Colab).